# RSNA Knee Abnormality Detection — Max-Pooling Submission

Submission notebook for the max-pooling configuration: fifteen central-band
slices per plane, collapsed by taking each dimension's strongest response
across slices rather than its average.

This carries **no experimental scaffolding**. The evaluation notebook measures
and compares; this one fits a single configuration and writes a submission, so
a competition rerun spends its time on inference rather than on comparisons
whose answers are already recorded.

The configuration is not the project's reported baseline. It scored higher
out-of-fold, but the difference did not survive the decision rule fixed before
the comparison, so it is submitted as a deliberate second entry rather than as
a promotion.

## 1. Environment and Frozen Configuration

In [ ]:
import hashlib
import importlib.metadata
import importlib.util
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError("This notebook runs on Kaggle only.")

# Verify and install the pinned offline wheel BEFORE inserting the source
# path or importing knee_mri anywhere, so the import cannot silently pick up
# a different stratifier than the one this contract pins.
package_initializers = tuple(Path("/kaggle/input/datasets").rglob("knee_mri/__init__.py"))
if len(package_initializers) != 1:
    raise RuntimeError("Expected exactly one attached knee_mri source package.")
_src_root = package_initializers[0].parent.parent
_dataset_root = _src_root.parent

WHEEL_NAME = "iterative_stratification-0.1.9-py3-none-any.whl"
EXPECTED_SHA256 = "476f8deff6753fb1725612fe41e59cc2058f8f2524ae5d1ccee88eb8c8d3de80"

wheel_matches = tuple(_dataset_root.rglob(WHEEL_NAME))
if len(wheel_matches) != 1:
    raise RuntimeError("Expected exactly one pinned iterative-stratification wheel.")
wheel_path = wheel_matches[0]
if hashlib.sha256(wheel_path.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError("Pinned iterative-stratification wheel checksum mismatch.")

try:
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", str(wheel_path)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
except OSError:
    raise RuntimeError("Failed to launch the offline install for the pinned wheel.") from None
if install_result.returncode != 0:
    raise RuntimeError("Offline installation of the pinned wheel failed.")
if importlib.metadata.version("iterative-stratification") != "0.1.9":
    raise RuntimeError("Installed iterative-stratification version mismatch.")

# The processor statistics come from the vendored copy of the attached
# model's own preprocessor_config.json. There is deliberately no fallback:
# substituting remembered constants is the silent-wrongness this contract
# exists to prevent.
PROCESSOR_CONFIG_NAME = "dinov2-small-preprocessor_config.json"
processor_matches = tuple(_dataset_root.rglob(PROCESSOR_CONFIG_NAME))
if len(processor_matches) != 1:
    raise RuntimeError("Expected exactly one vendored DINOv2 processor config.")
PROCESSOR_CONFIG_PATH = processor_matches[0]

# Install the vendored DICOM codec plugins. The corpus-wide census found no
# compressed series in either released split, so these are insurance for the
# hidden set rather than a current requirement -- but an undecodable slice
# there would fail silently into the fallback row, which is the failure mode
# worth spending a few seconds to avoid.
#
# --no-deps is required, not stylistic: both compiled wheels declare
# numpy>=2.0,<3.0, and without it pip would try to resolve or replace the
# kernel's own numpy, offline and unasked.
CODEC_WHEELS = {
    "pylibjpeg-2.1.0-py3-none-any.whl":
        "25df9496a69e64e98c887fddee12a1271e275b5f74ba804f9bf98a08bb80993e",
    "pylibjpeg_openjpeg-2.5.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl":
        "a22fcb649ba9849209d8e43dba88632445a5941f0cd6765338b3652a4c686140",
    "pylibjpeg_libjpeg-2.4.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl":
        "01d950ef496476a9223e4966376cb88098fcf5c55a12a21f722d7b5f84daae43",
}

codec_paths = []
for codec_name, codec_sha256 in CODEC_WHEELS.items():
    matches = tuple(_dataset_root.rglob(codec_name))
    if len(matches) != 1:
        raise RuntimeError("Expected exactly one copy of each vendored codec wheel.")
    if hashlib.sha256(matches[0].read_bytes()).hexdigest() != codec_sha256:
        raise RuntimeError("Vendored codec wheel checksum mismatch.")
    codec_paths.append(str(matches[0]))

try:
    codec_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", *codec_paths],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
except OSError:
    raise RuntimeError("Failed to launch the offline install for the codec wheels.") from None
if codec_result.returncode != 0:
    raise RuntimeError("Offline installation of the codec wheels failed.")

sys.path.insert(0, str(_src_root))

DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

In [ ]:
import torch
from transformers import AutoModel

from knee_mri.dataset import split_labeled_studies, validated_plane_candidates
from knee_mri.image_model import (
    IMAGE_CLASSIFIER_C,
    build_image_classifier,
    cross_validate_image_model,
    fit_image_model,
    fold_signature,
)
from knee_mri.intensity import load_processor_statistics
from knee_mri.labels import LABEL_COLUMNS
from knee_mri.laterality import DOMINANCE_GATE, SeriesLateralityEvidence, study_laterality
from knee_mri.model_selection import select_multilabel_folds
from knee_mri.series_audit import audit_series
from knee_mri.slice_sampling import (
    MINIMUM_DECODED_SLICES,
    SLICE_SAMPLE_SIZE,
    select_plane_sample,
)
from knee_mri.study_features import (
    EMBEDDING_DIM,
    PLANES,
    STUDY_VECTOR_DIM,
    PlaneInput,
    build_study_features,
    max_pool,
    mean_pool,
)
from knee_mri.submission import build_submission

In [ ]:
IMAGE_MEAN, IMAGE_STD = load_processor_statistics(PROCESSOR_CONFIG_PATH)

# The configuration being submitted. Sampling density and the within-plane
# pooling operator both differ from the evaluation notebook's defaults, so
# they are declared here rather than left implicit at their call sites.
SUBMISSION_SAMPLE_SIZE = 15
SUBMISSION_POOL = max_pool

submitted_contract = pd.Series(
    {
        "Slices sampled per plane": SUBMISSION_SAMPLE_SIZE,
        "Within-plane pooling": "max over slices",
        "Across-plane pooling": "mean over present planes",
        "Minimum decoded slices per plane": MINIMUM_DECODED_SLICES,
        "Study vector dimensions": STUDY_VECTOR_DIM,
        "Laterality dominance gate": DOMINANCE_GATE,
        "Classifier C": IMAGE_CLASSIFIER_C,
        "Classifier penalty": build_image_classifier().estimator.penalty,
        "Classifier solver": build_image_classifier().estimator.solver,
        "Classifier class_weight": build_image_classifier().estimator.class_weight,
        "Fold seed": SEED,
        "Processor image_mean": str(IMAGE_MEAN),
        "Processor image_std": str(IMAGE_STD),
    },
    name="Value",
).to_frame()

display(submitted_contract)

**Interpretation:** the two values that make this notebook different from the
evaluation baseline — fifteen slices and max pooling — are read back here
alongside the settings they share, so the submitted contract can be checked at
a glance rather than inferred from call sites further down.

## 2. Frozen Encoder

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Expected a GPU-enabled kernel for the image baseline.")


def _find_dinov2_dir(root: Path) -> Path:
    for config_path in root.rglob("config.json"):
        try:
            config = json.loads(config_path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if config.get("model_type") == "dinov2":
            return config_path.parent
    raise RuntimeError("Could not find an attached DINOv2 model source.")


DEVICE = torch.device("cuda")
_dinov2_dir = _find_dinov2_dir(Path("/kaggle/input"))
dinov2 = AutoModel.from_pretrained(str(_dinov2_dir), local_files_only=True)
dinov2 = dinov2.to(DEVICE).eval()
for parameter in dinov2.parameters():
    parameter.requires_grad_(False)

cuda_major, cuda_minor = torch.cuda.get_device_capability(0)
GPU_COMPATIBLE = f"sm_{cuda_major}{cuda_minor}" in torch.cuda.get_arch_list()
if not GPU_COMPATIBLE:
    raise RuntimeError("Allocated GPU compute capability unsupported by installed PyTorch.")


# Returns the CLS token and the mean of the patch tokens side by side, from a
# single forward pass. The reported baseline uses the CLS half; the
# pre-registered patch-pooling variant uses the other. Computing both here
# rather than in two passes is what guarantees they differ only in the
# representation and in nothing upstream of it.
WIDE_EMBEDDING_DIM = 2 * EMBEDDING_DIM


def encode_batch(batch: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        outputs = dinov2(pixel_values=batch.to(DEVICE), interpolate_pos_encoding=True)
    hidden = outputs.last_hidden_state
    cls_token = hidden[:, 0, :]
    patch_mean = hidden[:, 1:, :].mean(dim=1)
    return torch.cat([cls_token, patch_mean], dim=1).detach().cpu()


# Smoke test: a checksum proves the bytes, not that the plugin loads.
codec_plugins = {
    name: importlib.util.find_spec(name) is not None
    for name in ("pylibjpeg", "libjpeg", "openjpeg")
}
if not all(codec_plugins.values()):
    raise RuntimeError("A vendored codec plugin failed to import after install.")

environment_summary = pd.Series(
    {
        "torch version": importlib.metadata.version("torch"),
        "transformers version": importlib.metadata.version("transformers"),
        "CUDA device": torch.cuda.get_device_name(0),
        "CUDA compute capability": f"{cuda_major}.{cuda_minor}",
        "DINOv2 parameters": sum(p.numel() for p in dinov2.parameters()),
        "Codec plugins importable": str(sorted(codec_plugins)),
        "Encoder trainable parameters": sum(
            p.numel() for p in dinov2.parameters() if p.requires_grad
        ),
    },
    name="Value",
).to_frame()

display(environment_summary)

**Interpretation:** records the exact runtime this submission was produced on.
The compute capability matters in particular: an unpinned accelerator can land
on a card the installed PyTorch cannot use, and the run fails here rather than
part-way through inference.

## 3. Feature Extraction

The evaluation notebook's pipeline unchanged, with the sampling density and
the pooling operator set to the configuration being submitted.

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_series_df = pd.read_csv(DATA_DIR / "train_series.csv")
test_series_df = pd.read_csv(DATA_DIR / "test_series.csv")

labeled_studies, _ = split_labeled_studies(train_df)
labeled_studies = labeled_studies.reset_index(drop=True)

# Aggregate-only telemetry. Every entry is a count or a rate; no study or
# series identifier is ever placed in these structures.
telemetry = {
    "planes_absent": 0,
    "plane_retries": 0,
    "candidates_tried": 0,
    "decoded_slice_counts": [],
    "laterality_unreliable_studies": 0,
    "studies_with_no_plane": 0,
    "header_read_failures": 0,
}


def _study_laterality(series_df: pd.DataFrame, series_root: Path, study_id: str, sink=None):
    """Conservative consensus over EVERY available series in the study."""
    sink = telemetry if sink is None else sink
    evidence = []
    study_dir = series_root / study_id
    if not study_dir.is_dir():
        return study_laterality([])
    for series_dir in sorted(p for p in study_dir.iterdir() if p.is_dir()):
        try:
            audit = audit_series(series_dir, decode_sample_size=1)
        except FileNotFoundError:
            continue
        sink["header_read_failures"] += audit.header_read_failures
        evidence.append(
            SeriesLateralityEvidence(
                tag=audit.laterality_tag,
                geometry=audit.laterality_from_geometry,
                cross_tag_conflict=audit.laterality_cross_tag_conflict,
            )
        )
    return study_laterality(evidence)


def build_features_for(
    series_df: pd.DataFrame,
    series_root: Path,
    study_ids,
    timings=None,
    per_plane=None,
    sample_size: int = SLICE_SAMPLE_SIZE,
    sink=None,
    slice_pool=mean_pool,
    pooled_width: int = EMBEDDING_DIM,
) -> np.ndarray:
    # A second extraction pass at a different density must not fold its
    # counters into the baseline's, or the reported telemetry becomes a
    # mixture of two contracts. The sink defaults to the baseline's.
    sink = telemetry if sink is None else sink
    vectors = []
    for study_id in study_ids:
        study_start = time.perf_counter()
        study_dir = series_root / study_id
        # Total .dcm files in the study is the real I/O surface: ordering
        # validation reads every header of every candidate series, not just
        # the five slices ultimately decoded.
        study_slices = len(list(study_dir.rglob("*.dcm"))) if study_dir.is_dir() else 0
        planes = {}
        for plane in PLANES:
            candidates = validated_plane_candidates(series_df, series_root, study_id, plane)
            sink["candidates_tried"] += len(candidates)
            outcome = select_plane_sample(
                [paths for _, paths in candidates], sample_size=sample_size
            )
            if outcome.absent or outcome.sample is None:
                sink["planes_absent"] += 1
                continue
            if outcome.candidates_tried > 1:
                sink["plane_retries"] += 1
            sink["decoded_slice_counts"].append(outcome.sample.decoded)
            winning_paths = candidates[outcome.candidates_tried - 1][1]
            header = pydicom.dcmread(winning_paths[0], stop_before_pixels=True)
            planes[plane] = PlaneInput(
                images=outcome.sample.images,
                image_orientation_patient=[float(v) for v in header.ImageOrientationPatient],
                pixel_spacing=[float(v) for v in header.PixelSpacing],
            )

        features = build_study_features(
            planes,
            _study_laterality(series_df, series_root, study_id, sink=sink),
            encode_batch,
            IMAGE_MEAN,
            IMAGE_STD,
            embedding_dim=WIDE_EMBEDDING_DIM,
            slice_pool=slice_pool,
        )
        if not features.laterality_reliable:
            sink["laterality_unreliable_studies"] += 1
        if not features.has_any_plane:
            sink["studies_with_no_plane"] += 1
        if timings is not None:
            timings.append(
                {
                    "slices": study_slices,
                    "seconds": time.perf_counter() - study_start,
                }
            )
        if per_plane is not None:
            per_plane.append(features.plane_embeddings)
        # Slice the reported baseline back out of the wide vector: the CLS
        # half plus the four flags, exactly the frozen 388-wide contract.
        vectors.append(
            np.concatenate(
                [
                    features.vector[:pooled_width],
                    features.vector[WIDE_EMBEDDING_DIM:],
                ]
            )
        )
    return np.vstack(vectors)

import pydicom  # noqa: E402  (imported after the source path is established)

train_features = pd.DataFrame(
    build_features_for(
        train_series_df,
        DATA_DIR / "train_series",
        labeled_studies["StudyInstanceUID"],
        sample_size=SUBMISSION_SAMPLE_SIZE,
        slice_pool=SUBMISSION_POOL,
    )
)

extraction_telemetry = pd.Series(
    {
        "Labeled studies": float(len(labeled_studies)),
        "Planes absent": float(telemetry["planes_absent"]),
        "Decoded slices per plane (mean)": float(
            np.mean(telemetry["decoded_slice_counts"])
        ),
        "Studies with unreliable laterality": float(
            telemetry["laterality_unreliable_studies"]
        ),
        "Studies with no usable plane": float(telemetry["studies_with_no_plane"]),
    },
    name="Value",
).to_frame()

display(extraction_telemetry)

**Interpretation:** aggregate counts only, never an identifier. The decoded
slices per plane is the figure to check — it should sit near fifteen rather
than five, and a value near five would mean the density argument silently
failed to take effect.

## 4. Out-of-Fold Check

The same folds and the same head as the evaluation notebook, run once. This is
not a new measurement: it must reproduce the score already recorded for this
configuration, and fails the run if it does not.

In [ ]:
y = labeled_studies[LABEL_COLUMNS].astype(int).reset_index(drop=True)
selected_splits, folds = select_multilabel_folds(y, seed=SEED)
cv_result = cross_validate_image_model(train_features, y, folds)

# This configuration was already measured. Reproducing it bit-identically here
# is what proves the submitted model is the one that was measured, rather than
# something assembled to resemble it.
EXPECTED_MACRO_AUC = 0.6507039913
if abs(cv_result.pooled_macro_auc - EXPECTED_MACRO_AUC) > 1e-9:
    raise RuntimeError("submitted configuration does not reproduce its measured score")

oof_summary = pd.Series(
    {
        "Pooled OOF macro AUC": cv_result.pooled_macro_auc,
        "Selected fold count": float(selected_splits),
        "Fold assignment signature": fold_signature(
            labeled_studies["StudyInstanceUID"].tolist(), folds
        ),
    },
    name="Value",
).to_frame()

display(oof_summary)

**Interpretation:** this is a reproduction check, not a new measurement. The
score was established when this configuration was compared against the mean;
reproducing it bit-identically is what proves the model being submitted is the
one that was measured. A mismatch fails the run rather than shipping a
submission whose claimed score belongs to something else.

## 5. Refit and Test Inference

The refit sees every labelled row and must therefore never produce a reported
score — it exists only to predict the test studies.

In [ ]:
scaler, classifier = fit_image_model(train_features, y)

inference_start = time.perf_counter()
test_features = pd.DataFrame(
    build_features_for(
        test_series_df,
        DATA_DIR / "test_series",
        test_df["StudyInstanceUID"],
        sample_size=SUBMISSION_SAMPLE_SIZE,
        slice_pool=SUBMISSION_POOL,
    )
)
test_probabilities = classifier.predict_proba(scaler.transform(test_features.to_numpy()))
inference_seconds = time.perf_counter() - inference_start

submission = build_submission(sample_df, test_df["StudyInstanceUID"], test_probabilities)
submission.to_csv("/kaggle/working/submission.csv", index=False)

submission_summary = pd.Series(
    {
        "Submission rows": float(len(submission)),
        "Submission columns": float(submission.shape[1]),
        "Probability minimum": float(test_probabilities.min()),
        "Probability maximum": float(test_probabilities.max()),
        "Test inference seconds (visible studies)": inference_seconds,
        "Seconds per test study": inference_seconds / max(len(test_df), 1),
    },
    name="Value",
).to_frame()

display(submission_summary)

with open("/kaggle/working/maxpool_submission_summary.json", "w") as handle:
    json.dump(
        {
            "submitted_contract": json.loads(submitted_contract.to_json()),
            "extraction_telemetry": json.loads(extraction_telemetry.to_json()),
            "oof": json.loads(oof_summary.to_json()),
            "submission_shape": json.loads(submission_summary.to_json()),
        },
        handle,
        indent=2,
    )

**Interpretation:** the per-study inference cost is the number that matters for
a rerun against the full hidden test set, since this configuration decodes three
times as many slices per plane as the reported baseline. It is recorded so the
cost of the denser sampling is visible rather than assumed.